In [1]:
pip install sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import os
from sqlalchemy import create_engine

In [6]:
from sqlalchemy.engine import URL
url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="YOUR_PASSWORD",
    host="localhost",
    database="vendor_analysis"
)

engine = create_engine(url)

In [7]:
with engine.connect() as conn:
    print("Connected Successfully!")

Connected Successfully!


In [18]:

import time
# Function for small CSV files
def ingest_db(df, table_name, engine):
    df.to_sql(
        table_name,
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=50000,
        method='multi'
    )

# Function for large CSV files
def ingest_large_csv(file_path, table_name, engine, read_chunksize=100000):

    first_chunk = True

    for chunk in pd.read_csv(file_path, chunksize=read_chunksize):

        chunk.to_sql(
            table_name,
            con=engine,
            if_exists='replace' if first_chunk else 'append',
            index=False,
            method='multi',
            chunksize=2000
        )

        print(f"Inserted {len(chunk):,} rows into {table_name}")

        first_chunk = False

    print(f"✅ {table_name} imported successfully!\n")


# Folder containing all CSV files
folder = r"DATA_FOLDER_PATH"

# Files to import in order
files = [
    "purchase_prices.csv",
    "vendor_invoice.csv",
    "begin_inventory.csv",
    "end_inventory.csv",
    "purchases.csv",
    "sales.csv"
]

large_files = ["purchases.csv", "sales.csv"]

start = time.time()

for file in files:

    file_path = os.path.join(folder, file)

    if file in large_files:

        print(f"\nImporting Large File: {file}")

        ingest_large_csv(
            file_path=file_path,
            table_name=file[:-4],
            engine=engine,
            read_chunksize=100000
        )

    else:

        df = pd.read_csv(file_path)

        print(f"{file}: {df.shape}")

        ingest_db(
            df=df,
            table_name=file[:-4],
            engine=engine
        )

        print(f"✅ {file[:-4]} imported successfully!\n")

end = time.time()

print("=" * 60)
print(f"All tables imported successfully!")
print(f"Total Time: {(end - start)/60:.2f} minutes")
print("=" * 60)

purchase_prices.csv: (12261, 9)
✅ purchase_prices imported successfully!

vendor_invoice.csv: (5543, 10)
✅ vendor_invoice imported successfully!

begin_inventory.csv: (206529, 9)
✅ begin_inventory imported successfully!

end_inventory.csv: (224489, 9)
✅ end_inventory imported successfully!


Importing Large File: purchases.csv
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inserted 100,000 rows into purchases
Inser

In [19]:
import os
import pandas as pd

folder = r"DATA_FOLDER_PATH"

for file in sorted(os.listdir(folder)):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        print(f"{file:<25} {df.shape}")

begin_inventory.csv       (206529, 9)
end_inventory.csv         (224489, 9)
purchase_prices.csv       (12261, 9)
purchases.csv             (2372474, 16)
sales.csv                 (12825363, 14)
vendor_invoice.csv        (5543, 10)
